In [ ]:
from pathlib import Path
import glob
import sys

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / 'src' / 'obscalib').exists():
        ROOT = candidate
        break
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import torch
import matplotlib.pyplot as plt

from torch.utils.benchmark import Timer

from obscalib.geometry import (
    se3_exp,
    se3_hat,
    so3_exp,
    so3_hat,
)


device = "cuda" if torch.cuda.is_available() else "cpu"
# device = "cpu"
dtype = torch.float32

batch_sizes = [1, 16, 64, 256, 1024, 4096]

print(f"device: {device}")
print(f"dtype:  {dtype}")

In [ ]:
def benchmark_function(function, x, min_run_time=0.5):
    """Return median execution time in seconds."""

    timer = Timer(
        stmt="function(x)",
        globals={
            "function": function,
            "x": x,
        },
    )

    return timer.blocked_autorange(
        min_run_time=min_run_time
    ).median


def so3_matrix_exp(phi):
    """Reference SO(3) exponential using the generic matrix exponential."""

    return torch.matrix_exp(so3_hat(phi))


def se3_matrix_exp(xi):
    """Reference SE(3) exponential using the generic matrix exponential."""

    return torch.matrix_exp(se3_hat(xi))


# Check that both implementations produce the same transformations.
phi = 0.3 * torch.randn(1024, 3, device=device, dtype=dtype)
xi = 0.3 * torch.randn(1024, 6, device=device, dtype=dtype)

R_closed = so3_exp(phi)
R_matrix = so3_matrix_exp(phi)

T_closed = se3_exp(xi)
T_matrix = se3_matrix_exp(xi)

print(
    "SO(3) max absolute error:",
    (R_closed - R_matrix).abs().max().item(),
)

print(
    "SE(3) max absolute error:",
    (T_closed - T_matrix).abs().max().item(),
)

In [ ]:
results = []

for batch_size in batch_sizes:
    phi = 0.3 * torch.randn(
        batch_size,
        3,
        device=device,
        dtype=dtype,
    )

    xi = 0.3 * torch.randn(
        batch_size,
        6,
        device=device,
        dtype=dtype,
    )

    so3_closed_time = benchmark_function(
        so3_exp,
        phi,
    )
    so3_matrix_time = benchmark_function(
        so3_matrix_exp,
        phi,
    )

    se3_closed_time = benchmark_function(
        se3_exp,
        xi,
    )
    se3_matrix_time = benchmark_function(
        se3_matrix_exp,
        xi,
    )

    results.append(
        {
            "batch_size": batch_size,
            "so3_closed_ms": 1e3 * so3_closed_time,
            "so3_matrix_ms": 1e3 * so3_matrix_time,
            "so3_speedup": so3_matrix_time / so3_closed_time,
            "se3_closed_ms": 1e3 * se3_closed_time,
            "se3_matrix_ms": 1e3 * se3_matrix_time,
            "se3_speedup": se3_matrix_time / se3_closed_time,
        }
    )


for result in results:
    print(
        f"B={result['batch_size']:5d} | "
        f"SO3: {result['so3_speedup']:.2f}x | "
        f"SE3: {result['se3_speedup']:.2f}x"
    )

In [ ]:
batch_sizes_plot = [
    result["batch_size"]
    for result in results
]

so3_speedups = [
    result["so3_speedup"]
    for result in results
]

se3_speedups = [
    result["se3_speedup"]
    for result in results
]


plt.figure(figsize=(8, 5))

plt.plot(
    batch_sizes_plot,
    so3_speedups,
    marker="o",
    label="SO(3)",
)

plt.plot(
    batch_sizes_plot,
    se3_speedups,
    marker="o",
    label="SE(3)",
)

plt.axhline(
    1.0,
    linestyle="--",
)

plt.xscale("log")

plt.xlabel("Batch size")
plt.ylabel("Speedup over torch.matrix_exp")
plt.title(f"Closed-form Lie exponential speedup ({device}, {dtype})")

plt.grid(True)
plt.legend()
plt.show()